# P0 -- Fair GPU fine-tuning: Prithvi-EO-2.0 vs a from-scratch U-Net

**Run this on Kaggle** (Settings -> Accelerator -> GPU T4 x2 or P100; Settings -> Internet -> On).
Not executable/tested in this project's local CPU-only environment -- built directly
from verified, real API references (see the plan doc and the markdown notes below for
exactly what was checked and where), not from memory or guesswork. **If a cell errors on
first run, that is expected for a from-scratch Kaggle port -- bring the traceback back
and it gets fixed from there, not guessed around.**

See `Geospatial AI for Public Health -- AI Paper Track (Phases P0-P6) -- Plan.md` for
the full track this belongs to. This notebook is Phase P0 only: a fair Prithvi-vs-U-Net
comparison on H3/H7's existing 3-district Dhaka AOI. Phase P0b (Clay), P1 (new held-out
region), P2 (the real Sunamganj flood case study) are separate, later notebooks -- not
attempted here, so one untested piece at a time can actually be debugged.

**What makes this a *fair* comparison, unlike H7:**
- Both models see the **same 6-band input** (BLUE, GREEN, RED, NIR_NARROW, SWIR_1,
  SWIR_2) -- H7 compared Prithvi (which needs a specific band vocabulary) against a
  U-Net built on whatever features were at hand; here both get identical input.
- Both are a standard **3-class semantic segmentation** task (other / water / built-up,
  from the same WorldCover-derived labels), not H3/H7's two separate binary heads --
  one task, one loss family (dice), comparable per-class IoU for both models.
- Both train through the **same `pl.Trainer`**: same epoch budget, same optimizer
  (AdamW, lr=1e-4), same `ReduceLROnPlateau` schedule, same precision. H7 did not control
  for this.
- Prithvi's recipe (architecture, loss, optimizer, schedule, `freeze_backbone=False`)
  is copied from `blumenstiel/TerraTorch-Examples`' own maintained
  `prithvi_v2_eo_300_tl_unet_sen1floods11` example -- fetched and read directly
  (2026-10-01), not assumed. An earlier assumption in this project's planning chat (LR
  warmup + cosine decay) was **wrong** and is not used here.

## 1. Setup

In [ ]:
!pip install -q "numpy>=2" terratorch==0.99.8 lightning segmentation-models-pytorch pystac-client planetary-computer stackstac rioxarray
!python -c "import numpy, torch; print('numpy', numpy.__version__, '| torch', torch.__version__)"

!git clone -q https://github.com/khalilurrrahmanridoykhan/geohealth-risk-mapping.git repo
import sys
sys.path.insert(0, 'repo')


**Restart the kernel/session now** (Kaggle menu/toolbar -> restart, a real process restart, not just re-running a cell) before running the next cell.

Real reason, found running this on Kaggle (2026-10-01): the install above lets pip's resolver downgrade numpy (Kaggle's base image ships numpy 2.x; several packages here pull in a numpy<2 pin). Kaggle's kernel already has the old numpy loaded in memory from its own startup, so importing anything numpy-touching *without* a restart crashes with `ValueError: numpy.dtype size changed, may indicate binary incompatibility` -- a stale in-memory copy fighting the newly-written-to-disk version, not a real code bug. A full restart clears memory so only the post-install numpy ever gets loaded.

**If the import cell below still crashes with the same `numpy.dtype size changed` error after this restart**: the printed `numpy X | torch Y` line above tells us the real versions pip actually landed on -- paste that line plus the new traceback back, rather than restarting again, since a repeat crash after a clean restart means it's a genuine on-disk incompatible combination (most likely torch built against a different numpy ABI than what got installed), not a memory-caching issue a restart can fix on its own.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import lightning.pytorch as pl
import segmentation_models_pytorch as smp
import terratorch  # noqa: F401 -- registers backbones/decoders/necks
from torch.utils.data import DataLoader, Dataset

from src.imagery import get_imagery
from src.landcover import reclassify_worldcover, WATER, BUILT_UP
from src.tiling import tile_array, assign_tile_split

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
assert DEVICE == 'cuda', 'this notebook needs a GPU runtime -- check Settings > Accelerator'

## 2. Real data -- same AOI, same labels as H3/H7, now the full 6-band set

`bands=` is new (`src/imagery.py`, commit `1efd5a4`) -- H0-H9 only ever fetched 4 bands
(green/red/nir/swir); Prithvi's documented recipe needs blue and a second SWIR band too.

In [ ]:
import pystac_client, planetary_computer, stackstac

AOI = (90.30, 23.75, 90.55, 23.95)  # same 3-district tile as H0-H4, H7
PRITHVI_BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']  # BLUE GREEN RED NIR_NARROW* SWIR_1 SWIR_2
# *note: B08 is NIR_BROAD, not NIR_NARROW (B8A) -- same choice H0-H9 already made for
# the 4-band set (B08, not B8A); kept consistent rather than silently switching bands.

composite, meta = get_imagery(AOI, '2026-01-01/2026-02-28', sensor='sentinel-2',
                               cloud_cover_max=60, max_scenes=1, resolution=10,
                               bands=PRITHVI_BANDS)
print(meta, composite.shape)

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)
wc_search = catalog.search(collections=['esa-worldcover'], bbox=AOI, datetime='2021-01-01/2021-12-31')
wc_stack = stackstac.stack(list(wc_search.items()), assets=['map'], bounds_latlon=AOI, resolution=10, epsg=meta['epsg'])
worldcover = wc_stack.compute().isel(time=0, band=0).values
assert worldcover.shape == composite.shape[1:], 'grid mismatch'

labels_4class = reclassify_worldcover(worldcover)
# collapse to 3 mutually-exclusive classes for standard (not multi-label) segmentation:
# 0 = other (vegetation/bare), 1 = water, 2 = built_up
label_map = np.zeros_like(labels_4class, dtype='int64')
label_map[labels_4class == WATER] = 1
label_map[labels_4class == BUILT_UP] = 2
CLASS_NAMES = ['other', 'water', 'built_up']
print('class balance:', {n: float((label_map == i).mean()) for i, n in enumerate(CLASS_NAMES)})

## 3. Tile + spatial-block split (same convention as H3: eastern 30% held out)

In [ ]:
TILE_SIZE = 256

feat_tiles, coords = tile_array(np.nan_to_num(composite.values), tile_size=TILE_SIZE)
label_tiles, _ = tile_array(label_map, tile_size=TILE_SIZE)
test_flags = assign_tile_split(coords, width=composite.shape[-1], test_fraction=0.3)
print(f'{len(feat_tiles)} tiles total: {sum(not f for f in test_flags)} train, {sum(test_flags)} held-out (eastern block)')

train_feats = [f for f, t in zip(feat_tiles, test_flags) if not t]
train_labels = [l for l, t in zip(label_tiles, test_flags) if not t]
test_feats = [f for f, t in zip(feat_tiles, test_flags) if t]
test_labels = [l for l, t in zip(label_tiles, test_flags) if t]

# Prithvi's own recipe scales raw Sentinel-2 L2A digital numbers (0-10000-ish
# reflectance*1e4) by 1e-4 before standardizing -- same convention applied here.
CONSTANT_SCALE = 1e-4
scaled_train = np.stack(train_feats) * CONSTANT_SCALE
band_mean = scaled_train.mean(axis=(0, 2, 3))
band_std = scaled_train.std(axis=(0, 2, 3)) + 1e-6
print('per-band mean (post-scale):', band_mean.round(4))
print('per-band std  (post-scale):', band_std.round(4))


class TileDataset(Dataset):
    """Emits {'image': (C,H,W) float32, 'mask': (H,W) int64} -- the batch
    shape terratorch.tasks.SemanticSegmentationTask expects, so the exact
    same dataset/loader serves both the Prithvi task and the U-Net baseline."""
    def __init__(self, feats, labels):
        self.feats, self.labels = feats, labels

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, idx):
        x = (self.feats[idx] * CONSTANT_SCALE - band_mean[:, None, None]) / band_std[:, None, None]
        return {'image': torch.from_numpy(x.astype('float32')), 'mask': torch.from_numpy(self.labels[idx])}


BATCH_SIZE = 8
train_loader = DataLoader(TileDataset(train_feats, train_labels), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(TileDataset(test_feats, test_labels), batch_size=BATCH_SIZE, num_workers=2)
# TerraTorch's task wants a val_dataloader too; the held-out block doubles as both
# val (for ReduceLROnPlateau / checkpoint selection) and final test here, same as H3.
val_loader = test_loader

## 4. Prithvi, via TerraTorch -- the verified recipe

Copied from `blumenstiel/TerraTorch-Examples/configs/prithvi_v2_eo_300_tl_unet_sen1floods11.yaml`
and its companion notebook (fetched 2026-10-01), adapted only for: `num_classes=3` (not 2),
`backbone_img_size=256` (this project's tile size, not 512), and this AOI's 6-band order.
Everything else -- architecture, loss, optimizer, LR schedule, `freeze_backbone=False`,
`head_dropout` -- is the maintainer's own documented setting, not a guess.

In [ ]:
EPOCHS = 100  # the real config's value; GPU makes this cheap, unlike H7's CPU-bound 10

prithvi_model = terratorch.tasks.SemanticSegmentationTask(
    model_factory='EncoderDecoderFactory',
    model_args={
        'backbone': 'prithvi_eo_v2_300_tl',
        'backbone_pretrained': True,
        'backbone_img_size': TILE_SIZE,
        'backbone_bands': ['BLUE', 'GREEN', 'RED', 'NIR_BROAD', 'SWIR_1', 'SWIR_2'],
        'backbone_coords_encoding': [],
        'necks': [
            {'name': 'SelectIndices', 'indices': [5, 11, 17, 23]},  # prithvi_eo_v2_300's indices
            {'name': 'ReshapeTokensToImage'},
            {'name': 'LearnedInterpolateToPyramidal'},
        ],
        'decoder': 'UNetDecoder',
        'decoder_channels': [512, 256, 128, 64],
        'head_dropout': 0.1,
        'num_classes': 3,
    },
    loss='dice',
    optimizer='AdamW',
    lr=1e-4,
    freeze_backbone=False,  # the real CLI config's setting (full fine-tune)
    freeze_decoder=False,
    class_names=CLASS_NAMES,
)

prithvi_trainer = pl.Trainer(
    accelerator='gpu', devices=1, precision='16-mixed',
    max_epochs=EPOCHS, log_every_n_steps=5,
    callbacks=[pl.callbacks.EarlyStopping(monitor='val/loss', patience=15)],
    default_root_dir='output/p0_prithvi/',
)
prithvi_trainer.fit(prithvi_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
prithvi_test_metrics = prithvi_trainer.test(prithvi_model, dataloaders=test_loader)
print(prithvi_test_metrics)

## 5. U-Net baseline -- same input, same labels, same optimizer/schedule/epoch budget

Not a TerraTorch task, so it needs its own small Lightning wrapper -- but every training
hyperparameter it's given is deliberately copied from Prithvi's recipe above, not H3's
original choices, so a difference in the result reflects the *architectures*, not an
unmatched training budget (the gap this notebook exists to close from H7).

In [ ]:
class UNetBaseline(pl.LightningModule):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.model = smp.Unet(encoder_name='resnet34', encoder_weights=None,
                               in_channels=in_channels, classes=num_classes)
        self.dice = smp.losses.DiceLoss(mode='multiclass')
        self.num_classes = num_classes

    def forward(self, x):
        return self.model(x)

    def _step(self, batch, stage):
        logits = self(batch['image'])
        loss = self.dice(logits, batch['mask'])
        self.log(f'{stage}/loss', loss, prog_bar=(stage != 'train'))
        return logits, loss

    def training_step(self, batch, _):
        _, loss = self._step(batch, 'train')
        return loss

    def validation_step(self, batch, _):
        self._step(batch, 'val')

    def test_step(self, batch, _):
        logits, _ = self._step(batch, 'test')
        preds = logits.argmax(dim=1)
        for c, name in enumerate(CLASS_NAMES):
            p, t = preds == c, batch['mask'] == c
            union = (p | t).sum().float()
            iou = (p & t).sum().float() / union if union > 0 else torch.tensor(1.0, device=p.device)
            self.log(f'test/iou_{name}', iou)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=1e-4, weight_decay=0.1)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)
        return {'optimizer': optimizer, 'lr_scheduler': {'scheduler': scheduler, 'monitor': 'val/loss'}}


unet_model = UNetBaseline(in_channels=len(PRITHVI_BANDS), num_classes=3)
unet_trainer = pl.Trainer(
    accelerator='gpu', devices=1, precision='16-mixed',
    max_epochs=EPOCHS, log_every_n_steps=5,
    callbacks=[pl.callbacks.EarlyStopping(monitor='val/loss', patience=15)],
    default_root_dir='output/p0_unet/',
)
unet_trainer.fit(unet_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
unet_test_metrics = unet_trainer.test(unet_model, dataloaders=test_loader)
print(unet_test_metrics)

## 6. Compare, honestly

Both models' held-out per-class IoU, side by side. No seed averaging yet (that's P4) --
this is a single run to get the pipeline correct and both recipes verified end-to-end
before spending GPU hours on multiple seeds.

In [ ]:
import pandas as pd

def tidy(metrics, model_name):
    row = {'model': model_name}
    for k, v in metrics[0].items():
        if 'iou' in k:
            row[k.split('/')[-1]] = round(v, 3)
    return row

comparison = pd.DataFrame([
    tidy(prithvi_test_metrics, 'prithvi_eo_v2_300_tl'),
    tidy(unet_test_metrics, 'unet_resnet34_scratch'),
])
print(comparison.to_string(index=False))

## 7. Honest notes for whoever runs this next

- **This cell sequence has not been executed anywhere yet.** The APIs above are copied
  from verified, real, dated sources (see section headers), but Kaggle-specific friction
  (package version pins, `terratorch==0.99.8` compatibility with Kaggle's preinstalled
  CUDA/torch, memory limits on a T4) is expected on a first run and should be fixed from
  the actual traceback, not guessed at here.
- **`EarlyStopping(patience=15)` was added here**, not part of either verified recipe --
  a judgment call to avoid burning the full 100-epoch budget on both models during
  pipeline debugging. Worth reconsidering (or removing) once the pipeline is confirmed
  working, since the real config trains the full 100 epochs.
- **This notebook does not use `terratorch fit` with the raw YAML config** (the other
  documented path) -- the Python API was used instead so the exact same in-memory tiles
  and `pl.Trainer` config serve both the Prithvi task and the U-Net baseline. Functionally
  equivalent per TerraTorch's own docs, but worth knowing this is the less-traveled of
  the two paths if something behaves unexpectedly.
- Once this runs clean: bring back the actual numbers (not just "it worked") so P0b
  (Clay) and P1 (new held-out region) build on a real result, the same discipline every
  other phase in this project has followed.